# 03 Modelo Clasificacion Ticket Alto

Este notebook entrena el primer modelo del proyecto usando **Regresion Logistica** para clasificar si un ticket es alto o no.

## 1. Librerias

Se importan las librerias de modelado y evaluacion para la clasificacion.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay


## 2. Cargar la base EDA

Se usa el parquet de la etapa 02 como entrada del modelo.

In [ ]:
# Definir la ruta base del proyecto.
PROJECT_ROOT = Path.cwd().resolve().parents[1]

# Apuntar al parquet del EDA.
PARQUET_PATH = PROJECT_ROOT / "parquets" / "02_EDA_Base_Tickets" / "02_base_eda_tickets.parquet"

# Cargar la base para clasificacion.
df = pd.read_parquet(PARQUET_PATH)
df.head()

## 3. Seleccionar variables y target

Aqui se definen las variables usadas por la regresion logistica.

In [ ]:
# Definir las variables explicativas del modelo.
FEATURES = [
    "dia", "mes", "trimestre", "dia_semana", "dia_tipo", "fin_semana",
    "ciudad", "capacidad_sucursal", "tipo_empleado", "salario", "turno",
    "numero_mesa", "capacidad_mesa", "metodo_pago", "lineas_ticket",
    "cantidad_total", "platillos_distintos", "categorias_distintas",
    "incluye_bebida", "incluye_postre", "incluye_entrada", "incluye_platillo_fuerte"
]

# Definir la variable objetivo.
TARGET = "ticket_alto"

# Separar X y y para el entrenamiento.
X = df[FEATURES].copy()
y = df[TARGET].copy()

## 4. Particion de entrenamiento y prueba

Se divide la muestra para poder evaluar el modelo antes de entrenarlo con toda la base.

In [ ]:
# Dividir la base en entrenamiento y prueba con estratificacion.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Revisar dimensiones de las particiones.
X_train.shape, X_test.shape

## 5. Preprocesamiento y modelo

Se construye un pipeline con imputacion, escalado y codificacion categorica.

In [ ]:
# Separar columnas numericas y categoricas.
numericas = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categoricas = X.select_dtypes(include=["object"]).columns.tolist()

# Definir el preprocesador del pipeline.
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numericas,
        ),
        (
            "cat",
            Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categoricas,
        ),
    ]
)

# Definir el pipeline completo de clasificacion.
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced")),
])

# Entrenar el modelo.
pipeline.fit(X_train, y_train)

## 6. Evaluacion del modelo

Se calculan las metricas principales sobre la muestra de prueba.

In [ ]:
# Generar predicciones y probabilidades sobre test.
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

# Resumir metricas principales.
metricas = {
    "accuracy": round(float(accuracy_score(y_test, y_pred)), 4),
    "precision": round(float(precision_score(y_test, y_pred)), 4),
    "recall": round(float(recall_score(y_test, y_pred)), 4),
    "f1": round(float(f1_score(y_test, y_pred)), 4),
    "roc_auc": round(float(roc_auc_score(y_test, y_prob)), 4),
}

pd.DataFrame(metricas.items(), columns=["metrica", "valor"])

## 7. Revisar algunas predicciones

Esta vista permite validar rapidamente la salida del modelo.

In [ ]:
# Construir una vista rapida de resultados sobre test.
resultados_test = X_test.copy()
resultados_test["ticket_alto_real"] = y_test
resultados_test["ticket_alto_pred"] = y_pred
resultados_test["prob_ticket_alto"] = y_prob

# Mostrar las primeras filas de resultados.
resultados_test.head()

## 6. Visualizaciones del modelo

Estas graficas ayudan a interpretar mejor el comportamiento de la clasificacion.


In [ ]:
# Graficar la matriz de confusion del modelo en prueba.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Matriz de confusion")

# Graficar la distribucion de probabilidades para tickets altos y normales.
sns.histplot(
    data=resultados_test,
    x="prob_ticket_alto",
    hue="ticket_alto_real",
    bins=20,
    kde=True,
    stat="density",
    common_norm=False,
    ax=axes[1],
)
axes[1].set_title("Probabilidad estimada de ticket alto")
axes[1].set_xlabel("prob_ticket_alto")
plt.tight_layout()


In [ ]:
# Graficar la curva ROC para visualizar la separacion del modelo.
fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_predictions(y_test, y_prob, ax=ax)
ax.set_title("Curva ROC del modelo de clasificacion")
plt.tight_layout()
